## **Setup**

In [1]:
import os, sys

# Repository information
REPO_NAME = "RecSys-Challenge-2025"
REPO_URL  = f"github.com/Lv1g1/{REPO_NAME}.git"

# Detect environment
IS_COLAB = 'content' in os.getcwd()
IS_KAGGLE = 'kaggle' in os.getcwd()
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

WORKING_DIR = os.getcwd()

if IS_COLAB:
    WORKING_DIR = "/content"

    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

    # Get GitHub token via input
    def get_token():
        from getpass import getpass
        return getpass("GitHub Token: ")

elif IS_KAGGLE:
    WORKING_DIR = "/kaggle/working"

    # Get GitHub token from Kaggle secrets
    def get_token():
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret("Token")

# If local environment assume inside the repo
LOCAL_REPO_PATH = "/home/luigi/RecSys" if IS_LOCAL else os.path.join(WORKING_DIR, REPO_NAME)

# Clone the repository if it doesn't exist
if not os.path.exists(LOCAL_REPO_PATH):
    os.chdir(WORKING_DIR)
    token = get_token()

    !git clone https://{token}@{REPO_URL}
else:
    print("Repo already exists — pulling latest changes")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
    os.chdir(WORKING_DIR)

# Add to Python PATH
if LOCAL_REPO_PATH not in sys.path:
    sys.path.append(LOCAL_REPO_PATH)

if IS_COLAB:
    !pip install optuna

Repo already exists — pulling latest changes
Already up to date.


In [2]:
if IS_COLAB or False:  # Set to True if you want to recompile Cython files
    os.chdir(LOCAL_REPO_PATH)
    !python run_compile_all_cython.py
    os.chdir(WORKING_DIR)

In [3]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import scipy.sparse as sps
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
import gc

from Challenge.paths import load_holdout_split, XGBOOST_MODELS, XGBOOST_DATAFRAMES
from Challenge.utils import load_models

Running on local — storage at: /home/luigi/RecSys


## **Load Data**

In [4]:
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample

URM_train_complete, URM_test = load_holdout_split()
URM_train, URM_validation = split_train_in_two_percentage_global_sample(URM_train_complete, train_percentage = 0.8)

## **Recommeder List**

In [5]:
from Recommenders.NonPersonalizedRecommender import TopPop
from Recommenders.KNN.UserKNNCFRecommender import UserKNNCFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.GraphBased.P3alphaRecommender import P3alphaRecommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.SLIM.SLIMElasticNetRecommender import MultiThreadSLIM_SLIMElasticNetRecommender
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_WARP_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_BPR_Cython
from Recommenders.MatrixFactorization.Cython.MatrixFactorization_Cython import MatrixFactorization_SVDpp_Cython

from Recommenders.SLIM.Cython.SLIM_BPR_Cython import SLIM_BPR_Cython
from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender
from Recommenders.MatrixFactorization.NMFRecommender import NMFRecommender
from Recommenders.Neural.MultVAE_PyTorch_Recommender import MultVAERecommender_PyTorch_OptimizerMask

models_mapping = {
    'TopPop': TopPop,
    'ItemKNN_cosine': ItemKNNCFRecommender,
    'ItemKNN_jaccard': ItemKNNCFRecommender,
    'ItemKNN_asymmetric': ItemKNNCFRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'ItemKNN_dice': ItemKNNCFRecommender,
    'UserKNN_cosine': UserKNNCFRecommender,
    'UserKNN_jaccard': UserKNNCFRecommender,
    'UserKNN_asymmetric': UserKNNCFRecommender,
    'UserKNN_tversky': UserKNNCFRecommender,
    'UserKNN_dice': UserKNNCFRecommender,
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'EASE_R': EASE_R_Recommender,
    'P3alpha': P3alphaRecommender,
    'RP3beta': RP3betaRecommender,
    'IALS': IALSRecommender,
    'MatrixFactorization_WARP': MatrixFactorization_WARP_Cython,
    'MatrixFactorization_BPR': MatrixFactorization_BPR_Cython,
    'MatrixFactorization_SVDpp': MatrixFactorization_SVDpp_Cython,
    
    'SLIM_BPR': SLIM_BPR_Cython,
    'NMF': NMFRecommender,
    'MultVAE': MultVAERecommender_PyTorch_OptimizerMask,
}

candidate_mapping = {
    'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender,
    'ItemKNN_tversky': ItemKNNCFRecommender,
    'RP3beta': RP3betaRecommender,
    'TopPop': TopPop
}

candidate_cutoff = {
    'SLIMElasticNet': 100,
    'ItemKNN_tversky': 100,
    'RP3beta': 100,
    'TopPop': 100
}

In [6]:
# Train all the models

# model_folder = os.path.join(XGBOOST_MODELS, "train_candidates")
# models = load_models(URM_train, candidate_mapping, model_folder=model_folder)

# model_folder = os.path.join(XGBOOST_MODELS, "validation_candidates")
# models = load_models(URM_train_complete, candidate_mapping, model_folder=model_folder)

# model_folder = os.path.join(XGBOOST_MODELS, "pred_candidates")
# models = load_models(URM_train_complete + URM_test, candidate_mapping, model_folder=model_folder)

# model_folder = os.path.join(XGBOOST_MODELS, "train_models")
# models = load_models(URM_train, models_mapping, model_folder=model_folder)

# model_folder = os.path.join(XGBOOST_MODELS, "validation_models")
# models = load_models(URM_train_complete, models_mapping, model_folder=model_folder)

# model_folder = os.path.join(XGBOOST_MODELS, "pred_models")
# models = load_models(URM_train_complete + URM_test, models_mapping, model_folder=model_folder)

# [name for name, model in models]

## **Functions**

In [7]:
def generate_candidates(URM_train, model_folder):
    model_folder = os.path.join(XGBOOST_MODELS, model_folder)
    candidate_models = load_models(URM_train, candidate_mapping, model_folder=model_folder)

    n_users, n_items = URM_train.shape
    user_ids = np.arange(n_users)
    
    dfs_to_concat = []
    for model_name, recommender in candidate_models:
        cutoff = candidate_cutoff[model_name]
        print(f"Generating candidates for model: {model_name}")
        
        recommendations_model = recommender.recommend(user_ids, cutoff=cutoff)
        
        df_model = pd.DataFrame({
            "UserID": user_ids,
            "ItemID": recommendations_model
        })

        df_model = df_model.explode("ItemID")

        dfs_to_concat.append(df_model)

    
    # Concatenate
    print("Concatenating and removing duplicates...")
    training_dataframe = pd.concat(dfs_to_concat, ignore_index=True)
    
    # Ensure correct data types (Explode can sometimes create objects)
    training_dataframe["ItemID"] = training_dataframe["ItemID"].astype(int)
    
    # Drop duplicates
    training_dataframe = training_dataframe.drop_duplicates(subset=["UserID", "ItemID"])

    return training_dataframe

In [8]:
def get_user_batches(user_ids, batch_size=1000):
    for i in range(0, len(user_ids), batch_size):
        yield user_ids[i:i + batch_size]

def add_models_features(training_dataframe, URM_train, model_folder):
    model_folder = os.path.join(XGBOOST_MODELS, model_folder)
    models = load_models(URM_train, models_mapping, model_folder=model_folder)

    # Sort by UserID
    training_dataframe = training_dataframe.sort_values("UserID").reset_index(drop=True)
    
    # We need the values as numpy arrays for fast slicing
    cand_users = training_dataframe['UserID'].values
    cand_items = training_dataframe['ItemID'].values
    
    # Store indices to map back to the original dataframe order later
    original_indices = training_dataframe.index.values

    N_CANDIDATES = len(training_dataframe)
    unique_users = training_dataframe['UserID'].unique()
    BATCH_SIZE = 500 # Kept smaller to handle the Rank Matrix memory

    # Dictionary to hold the final result arrays
    # Key = Column Name, Value = Numpy Array of shape (N_CANDIDATES,)
    results_arrays = {}

    for label, recommender in models:
        print(f"Processing features for model: {label}")
        
        # Allocate arrays aligned with the SORTED feature_candidates
        scores_final = np.zeros(N_CANDIDATES, dtype=np.float32)
        ranks_final = np.zeros(N_CANDIDATES, dtype=np.int32)
        
        for user_batch in tqdm(list(get_user_batches(unique_users, BATCH_SIZE)), desc=f"Batches {label}", leave=False):
            
            # Compute Scores
            scores_batch = recommender._compute_item_score(user_id_array=user_batch)

            # Normalize
            # Use max(..., 1e-9) to avoid division by zero without adding constant everywhere
            norm_factor = np.linalg.norm(scores_batch, np.inf, axis=1, keepdims=True)
            norm_factor[norm_factor == 0] = 1.0 
            linf_scores_batch = scores_batch / norm_factor

            # Remove Seen
            for i, user_id in enumerate(user_batch):
                linf_scores_batch[i, :] = recommender._remove_seen_on_scores(user_id, linf_scores_batch[i, :])

            # Calculate Ranks
            # argsort sorts ascending, so we use [::-1] to get descending (highest score first)
            # This returns INDICES of items. 
            # shape: (batch_size, n_items)
            rank_order = np.argsort(linf_scores_batch, axis=1)[:, ::-1]
            
            # We need the inverse: Given ItemID, what is its Rank?
            n_batch, n_items = scores_batch.shape
            rank_matrix_batch = np.empty((n_batch, n_items), dtype=np.int32)
            
            # Fancy numpy trick to invert the permutation vectors in one go
            # Arrays of shape (n_batch, 1) needed for broadcasting
            row_indices = np.arange(n_batch)[:, None] 
            rank_matrix_batch[row_indices, rank_order] = np.arange(n_items)
            
            # Fast Lookup using SearchSorted (requires sorted cand_users)
            start_pos = np.searchsorted(cand_users, user_batch[0], side='left')
            end_pos = np.searchsorted(cand_users, user_batch[-1], side='right')
            
            # Slice the global candidate arrays
            batch_cand_items = cand_items[start_pos:end_pos]
            batch_cand_users = cand_users[start_pos:end_pos]
            
            # We need to map global UserIDs to local batch indices (0 to BATCH_SIZE-1)
            local_user_indices = np.searchsorted(user_batch, batch_cand_users)
            
            # Extract Values
            scores_final[start_pos:end_pos] = linf_scores_batch[local_user_indices, batch_cand_items]
            ranks_final[start_pos:end_pos] = rank_matrix_batch[local_user_indices, batch_cand_items]
            
            # Explicit cleanup
            del scores_batch, linf_scores_batch, rank_matrix_batch, rank_order

        # Store results
        results_arrays[f"{label}_Score"] = scores_final
        results_arrays[f"{label}_RankPosition"] = ranks_final
        results_arrays[f"{label}_Recommended"] = (ranks_final < 10).astype(int)
        
        gc.collect()

    # Create a DataFrame from the new columns
    new_features_df = pd.DataFrame(results_arrays)
    
    # Join columns. No index matching needed because they are perfectly aligned 0..N
    training_dataframe = pd.concat([training_dataframe, new_features_df], axis=1)
    
    return training_dataframe

In [9]:
def calculate_item_item_features_fast(training_dataframe, URM_train, model_folder):
    # Load only distinct model types
    similarity_models = load_models(
        URM_train,
        {
            'ItemKNN_tversky': ItemKNNCFRecommender, 
            'RP3beta': RP3betaRecommender,
            'SLIMElasticNet': MultiThreadSLIM_SLIMElasticNetRecommender 
        },
        model_folder=os.path.join(XGBOOST_MODELS, model_folder)
    )
    
    # Sort dataframe by UserID to allow fast slicing
    df_sorted = training_dataframe.reset_index()[['UserID', 'ItemID']].sort_values("UserID").copy()
    
    # Pre-calculate mapping: UserID -> [List of Candidate ItemIDs]
    # We use numpy split for max speed
    user_ids = df_sorted['UserID'].values
    item_ids = df_sorted['ItemID'].values
    
    # Find indices where user changes
    unique_users, user_starts = np.unique(user_ids, return_index=True)
    # Map UserID to (start_index, end_index) in the sorted arrays
    user_map = {}
    for i, user_id in enumerate(unique_users):
        start = user_starts[i]
        end = user_starts[i+1] if i + 1 < len(unique_users) else len(user_ids)
        user_map[user_id] = (start, end)
    
    # Prepare result dictionary
    N_ROWS = len(df_sorted)
    
    for label, recommender in similarity_models:
        print(f"Extracting MAX similarity for {label}...")
        
        # Get Sparse Matrix
        W_sparse = recommender.W_sparse
        if not sps.issparse(W_sparse):
            W_sparse = sps.csr_matrix(W_sparse)
            
        # Feature arrays
        max_sims = np.zeros(N_ROWS, dtype=np.float32)
        std_sims = np.zeros(N_ROWS, dtype=np.float32)
        
        # Iterate over unique users in the candidates
        for user_id in tqdm(unique_users):
            start, end = user_map[user_id]
            
            # Get Seen Items for this user (Indices)
            seen_items = URM_train.indices[URM_train.indptr[user_id]:URM_train.indptr[user_id+1]]
            
            if len(seen_items) == 0:
                continue
                
            # Get Candidate Items for this user
            cand_items = item_ids[start:end]
            
            # --- THE CORE OPTIMIZATION ---
            # Instead of .toarray(), we slice sparse matrix
            # Submatrix: Rows=Candidates, Cols=SeenItems
            # This is efficient because W is CSR (fast row slicing)
            sub_W = W_sparse[cand_items, :][:, seen_items]
            
            # Check if sub_W is effectively empty
            if sub_W.nnz > 0:
                # Max similarity to any seen item
                max_sims[start:end] = sub_W.max(axis=1).toarray().flatten()
                
                # For Std, you usually need to densify (slower)
                dense_batch = sub_W.toarray()
                std_sims[start:end] = dense_batch.std(axis=1)

        # Assign to dataframe
        df_sorted[f'{label}_MaxSim'] = max_sims
        df_sorted[f'{label}_StdSim'] = std_sims

    # If UserID is the index, move it to a column.
    if training_dataframe.index.name == 'UserID':
        training_dataframe = training_dataframe.reset_index()
        
    # Merge
    training_dataframe = training_dataframe.merge(
        df_sorted, on=['UserID', 'ItemID'], how='left'
    )
    
    # Set index back to UserID
    training_dataframe = training_dataframe.set_index("UserID")
    
    return training_dataframe

In [10]:
def add_embedding_features_batched(training_dataframe, URM_train, model_folder, 
                                   pca_components=5, n_item_clusters=10, n_user_clusters=10, 
                                   batch_size=200000):
    
    # 1. Handle Index (Safety First)
    if training_dataframe.index.name == 'UserID':
        print("Resetting index to restore UserID column...")
        training_dataframe = training_dataframe.reset_index()

    # 2. Load Model & Factors
    print("Loading IALS model...")
    model_path = os.path.join(XGBOOST_MODELS, model_folder)
    recommender = IALSRecommender(URM_train)
    recommender.load_model(model_path, "IALS.zip")
    
    user_factors = recommender.USER_factors.astype(np.float32)
    item_factors = recommender.ITEM_factors.astype(np.float32)
    
    # --- PART A: CLUSTERING (Low Memory, do it all at once) ---
    print("Calculating K-Means Clusters...")
    
    # Cluster Users
    kmeans_users = KMeans(n_clusters=n_user_clusters, random_state=42, n_init=10).fit(user_factors)
    # Map directly using pandas map (fast)
    training_dataframe['User_Cluster'] = kmeans_users.labels_[training_dataframe['UserID'].values.astype(int)]
    
    # Cluster Items
    kmeans_items = KMeans(n_clusters=n_item_clusters, random_state=42, n_init=10).fit(item_factors)
    training_dataframe['Item_Cluster'] = kmeans_items.labels_[training_dataframe['ItemID'].values.astype(int)]
    
    del kmeans_users, kmeans_items
    gc.collect()

    # --- PART B: PCA SETUP (Fit on a small random sample) ---
    print("Fitting PCA on random sample...")
    
    # Pick 50k random indices to learn the PCA transformation
    # We do this OUTSIDE the loop so the "Coordinate System" is consistent
    n_samples = min(50000, len(training_dataframe))
    sample_indices = np.random.choice(len(training_dataframe), n_samples, replace=False)
    
    sample_users = training_dataframe['UserID'].values[sample_indices].astype(int)
    sample_items = training_dataframe['ItemID'].values[sample_indices].astype(int)
    
    sample_interactions = user_factors[sample_users] * item_factors[sample_items]
    
    pca = PCA(n_components=pca_components)
    pca.fit(sample_interactions)
    
    del sample_interactions, sample_users, sample_items
    gc.collect()

    # --- PART C: BATCHED CALCULATION (Euclidean & PCA Transform) ---
    print(f"Processing Batches (Batch Size: {batch_size})...")
    
    num_rows = len(training_dataframe)
    
    # 1. Pre-allocate columns with float32 (saves 50% RAM compared to float64)
    # We initialize with zeros
    training_dataframe['IALS_EuclideanDist'] = np.zeros(num_rows, dtype=np.float32)
    for i in range(pca_components):
        training_dataframe[f'IALS_PCA_{i}'] = np.zeros(num_rows, dtype=np.float32)
    
    # 2. Get the numpy array views of the dataframe columns for fast writing
    # (Writing to these arrays updates the dataframe directly)
    user_ids_all = training_dataframe['UserID'].values.astype(int)
    item_ids_all = training_dataframe['ItemID'].values.astype(int)
    
    dist_column = training_dataframe['IALS_EuclideanDist'].values
    pca_columns = [training_dataframe[f'IALS_PCA_{i}'].values for i in range(pca_components)]
    
    # 3. The Batch Loop
    for start_idx in tqdm(range(0, num_rows, batch_size)):
        end_idx = min(start_idx + batch_size, num_rows)
        
        # A. Get Vectors for this batch only
        batch_u_idx = user_ids_all[start_idx:end_idx]
        batch_i_idx = item_ids_all[start_idx:end_idx]
        
        batch_u_vecs = user_factors[batch_u_idx]
        batch_i_vecs = item_factors[batch_i_idx]
        
        # B. Euclidean Distance
        # Vectorized batch calculation
        diff = batch_u_vecs - batch_i_vecs
        batch_dist = np.linalg.norm(diff, axis=1)
        
        # Assign to main array
        dist_column[start_idx:end_idx] = batch_dist
        
        # C. PCA Transform
        # Calculate interaction for batch
        batch_interaction = batch_u_vecs * batch_i_vecs
        
        # Transform using the pre-fitted PCA
        batch_pca = pca.transform(batch_interaction)
        
        # Assign columns
        for k in range(pca_components):
            pca_columns[k][start_idx:end_idx] = batch_pca[:, k]
            
        # D. Cleanup
        del batch_u_vecs, batch_i_vecs, diff, batch_interaction, batch_pca, batch_dist
        # Optional: gc.collect() every few batches if RAM is extremely tight
        # if start_idx % (batch_size * 5) == 0: gc.collect()

    print("Embedding features added successfully.")
    return training_dataframe

In [11]:
def add_aggregate_features_stats(training_dataframe):
    # Consensus Features
    recommended_columns = [col for col in training_dataframe.columns if col.endswith('_Recommended')]
    training_dataframe['Counter_Recommended'] = training_dataframe[recommended_columns].sum(axis=1).astype(int)

    # Rank Position Statistics
    position_columns = [col for col in training_dataframe.columns if col.endswith('_RankPosition')]

    training_dataframe['Mean_RankPosition'] = training_dataframe[position_columns].mean(axis=1)
    training_dataframe['Std_RankPosition'] = training_dataframe[position_columns].std(axis=1)
    training_dataframe['Skew_RankPosition'] = training_dataframe[position_columns].skew(axis=1)
    training_dataframe['Kurtosis_RankPosition'] = training_dataframe[position_columns].kurtosis(axis=1)

    # Score Statistics
    score_columns = [col for col in training_dataframe.columns if col.endswith('_Score')]

    training_dataframe['Mean_Score'] = training_dataframe[score_columns].mean(axis=1)
    training_dataframe['Std_Score'] = training_dataframe[score_columns].std(axis=1)
    training_dataframe['Skew_Score'] = training_dataframe[score_columns].skew(axis=1)
    training_dataframe['Kurtosis_Score'] = training_dataframe[score_columns].kurtosis(axis=1)
    
    return training_dataframe

In [12]:
def add_user_stats(training_dataframe):
    # We need to map UserID and ItemID to the URM indices
    user_ids = training_dataframe['UserID'].values
    item_ids = training_dataframe['ItemID'].values
    
    # User Profile Length
    user_profile_len = np.ediff1d(URM_train.indptr)
    training_dataframe['User_Profile_Len'] = user_profile_len[user_ids]
    
    # Item Global Popularity
    item_popularity = np.ediff1d(URM_train.tocsc().indptr)
    training_dataframe['Item_Global_Popularity'] = item_popularity[item_ids]

    return training_dataframe

In [13]:
def sanity_check(df, verbose=True):
    print("--- STARTING SANITY CHECK ---")
    problems_found = False
    
    # 1. Check for Missing Values (NaN)
    null_counts = df.isnull().sum()
    if null_counts.sum() > 0:
        print("\n[CRITICAL] NaN Values Found:")
        print(null_counts[null_counts > 0])
        problems_found = True
    else:
        if verbose: print("[OK] No NaNs found.")

    # 2. Check for Infinite Values (inf / -inf)
    # Common issue when normalizing by zero variance or dividing scores
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    inf_counts = np.isinf(df[numeric_cols]).sum()
    if inf_counts.sum() > 0:
        print("\n[CRITICAL] Infinite Values Found (Division by Zero?):")
        print(inf_counts[inf_counts > 0])
        problems_found = True
    else:
        if verbose: print("[OK] No Infinite values found.")

    # 3. Check for Duplicates (UserID, ItemID)
    # Stacking fails if you have multiple rows for the same User-Item pair
    if df.duplicated(subset=['UserID', 'ItemID']).any():
        n_dupes = df.duplicated(subset=['UserID', 'ItemID']).sum()
        print(f"\n[CRITICAL] Duplicate (UserID, ItemID) pairs found: {n_dupes}")
        problems_found = True
    else:
        if verbose: print("[OK] Keys (UserID, ItemID) are unique.")

    # 4. Check Data Types (IDs must be int)
    # Merges (pd.merge) often convert ints to float if there were missing keys initially
    if df['UserID'].dtype not in [int, np.int32, np.int64]:
        print(f"\n[WARNING] UserID is {df['UserID'].dtype}, expected int. (Did a merge fail?)")
        # Auto-fix attempt
        # df['UserID'] = df['UserID'].astype(int) 
    
    if df['ItemID'].dtype not in [int, np.int32, np.int64]:
        print(f"\n[WARNING] ItemID is {df['ItemID'].dtype}, expected int.")

    # 5. Check for Constant Columns (Zero Variance)
    # These crash some implementations of Normalization and add no info to XGBoost
    std_devs = df[numeric_cols].std()
    constant_cols = std_devs[std_devs == 0].index.tolist()
    if len(constant_cols) > 0:
        print("\n[WARNING] The following columns have ZERO variance (Constant values):")
        print(constant_cols)
        print("Recommendation: Drop them.")
    
    # 6. Check Logic (Ranks shouldn't be negative)
    rank_cols = [c for c in df.columns if 'Rank' in c and 'Skew' not in c and 'Kurtosis' not in c]
    if rank_cols:
        min_ranks = df[rank_cols].min()
        if (min_ranks < 0).any():
             print("\n[CRITICAL] Negative Ranks found (Logic Error):")
             print(min_ranks[min_ranks < 0])
             problems_found = True

    if problems_found:
        print("\n--- SANITY CHECK FAILED: Fix errors before training ---")
        # raise ValueError("Data Integrity Issues Found") # Uncomment to force stop
    else:
        print("\n--- SANITY CHECK PASSED: Data is clean ---")

    return not problems_found

In [14]:
def optimize_dataframe_types(df):
    # 1. Downcast Integers (UserID, ItemID, Ranks)
    # int64 -> int32 (saves 50% RAM)
    ints = df.select_dtypes(include=['int64', 'int32', 'int']).columns
    df[ints] = df[ints].apply(pd.to_numeric, downcast='integer')
    
    # 2. Downcast Floats (Scores, Similarities)
    # float64 -> float32 (saves 50% RAM, XGBoost doesn't need float64)
    floats = df.select_dtypes(include=['float64', 'float']).columns
    df[floats] = df[floats].apply(pd.to_numeric, downcast='float')
    
    return df

## **Training Dataframe**

### **Generate Candidates**

In [15]:
training_dataframe = generate_candidates(URM_train, "train_candidates")
training_dataframe

Model found: SLIMElasticNet
Model found: ItemKNN_tversky
Model found: RP3beta
Model found: TopPop
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_candidatesSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Generating candidates for model: SLIMElasticNet
Unloading SLIMElasticNet...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_candidatesItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Generating candidates for model: ItemKNN_tversky
Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_candidatesRP3beta'
RP3betaRecommender: Loading complete
Generating candidates for model: RP3beta
Unloading RP3beta...
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_candidatesTopPop'
TopPopRecommender: Loading complet

,UserID,ItemID
0,0,2530
1,0,2194
2,0,2226
3,0,4264
4,0,2392
...,...,...
10837991,27094,4912
10837992,27094,3966
10837993,27094,3604
10837996,27094,4796


In [16]:
URM_validation_coo = sps.coo_matrix(URM_validation)

correct_recommendations = pd.DataFrame({"UserID": URM_validation_coo.row,
                                        "ItemID": URM_validation_coo.col})
correct_recommendations

,UserID,ItemID
0,0,325
1,0,1278
2,0,1339
3,0,1386
4,0,1727
...,...,...
486884,27094,6637
486885,27094,6763
486886,27094,6853
486887,27094,6856


In [17]:
# Label the training dataframe
training_dataframe = pd.merge(training_dataframe, correct_recommendations, on=['UserID','ItemID'], how='left', indicator='Exist')
training_dataframe["Label"] = training_dataframe["Exist"] == "both"
training_dataframe.drop(columns = ['Exist'], inplace=True)
training_dataframe

,UserID,ItemID,Label
0,0,2530,False
1,0,2194,True
2,0,2226,True
3,0,4264,False
4,0,2392,False
...,...,...,...
6601559,27094,4912,False
6601560,27094,3966,False
6601561,27094,3604,False
6601562,27094,4796,False


In [18]:
training_dataframe.Label.sum()

np.int64(281505)

### **Models Features**

In [19]:
training_dataframe = add_models_features(training_dataframe, URM_train, "train_models")
training_dataframe

Model found: TopPop
Model found: ItemKNN_cosine
Model found: ItemKNN_jaccard
Model found: ItemKNN_asymmetric
Model found: ItemKNN_tversky
Model found: ItemKNN_dice
Model found: UserKNN_cosine
Model found: UserKNN_jaccard
Model found: UserKNN_asymmetric
Model found: UserKNN_tversky
Model found: UserKNN_dice
Model found: SLIMElasticNet
Model found: EASE_R
Model found: P3alpha
Model found: RP3beta
Model found: IALS
Model found: MatrixFactorization_WARP
Model found: MatrixFactorization_BPR
Model found: MatrixFactorization_SVDpp
Model found: SLIM_BPR
Model found: NMF
Model found: MultVAE
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsTopPop'
TopPopRecommender: Loading complete
Processing features for model: TopPop


Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_cosine


Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_jaccard


Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsItemKNN_asymmetric'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_asymmetric


Unloading ItemKNN_asymmetric...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_tversky


...

Unloading SLIM_BPR...
Loading NMF...
NMFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsNMF'
NMFRecommender: Loading complete
Processing features for model: NMF


Unloading NMF...
Loading MultVAE...
Processing features for model: MultVAE


Batches MultVAE:   0%|          | 0/55 [00:00<?, ?it/s]/home/luigi/RecSys/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:203: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(


Unloading MultVAE...


,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,...,MatrixFactorization_SVDpp_Recommended,SLIM_BPR_Score,SLIM_BPR_RankPosition,SLIM_BPR_Recommended,NMF_Score,NMF_RankPosition,NMF_Recommended,MultVAE_Score,MultVAE_RankPosition,MultVAE_Recommended
0,0,3607,False,0.454812,26,0,0.699813,12,0,0.585087,...,0,0.743486,10,0,1.790960e-01,28,0,0.654270,105,0
1,0,3551,False,0.155081,313,0,0.510249,82,0,0.112332,...,0,0.233716,171,0,1.213270e-01,54,0,0.776577,21,0
2,0,6730,False,0.366663,55,0,0.568895,48,0,0.199446,...,0,0.561192,34,0,2.900852e-01,12,0,0.714261,52,0
3,0,6492,False,0.063416,935,0,0.426659,144,0,0.106174,...,0,0.031516,502,0,5.078765e-02,181,0,0.695365,67,0
4,0,3754,False,0.072911,818,0,0.386167,194,0,0.150095,...,0,0.108959,281,0,9.306875e-02,83,0,0.594878,154,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6601559,27094,6330,False,0.410620,35,0,0.048795,1800,0,0.000000,...,0,0.149295,174,0,0.000000e+00,6646,0,-0.149982,3571,0
6601560,27094,6584,False,0.412144,34,0,0.104053,997,0,0.000000,...,0,0.181066,143,0,1.243114e-06,4865,0,0.000014,2513,0
6601561,27094,2369,False,0.428672,29,0,0.092265,1110,0,0.000000,...,0,0.214552,123,0,2.918561e-01,55,0,0.012532,2433,0
6601562,27094,3316,False,0.424921,30,0,0.050882,1757,0,0.000000,...,0,0.162058,157,0,6.200765e-02,627,0,-0.273914,4493,0


In [20]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Item Similarity Features**

In [21]:
training_dataframe = calculate_item_item_features_fast(training_dataframe, URM_train, "train_models")
training_dataframe

Model found: ItemKNN_tversky
Model found: RP3beta
Model found: SLIMElasticNet
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Extracting MAX similarity for ItemKNN_tversky...


100%|██████████| 27095/27095 [00:04<00:00, 5595.60it/s]


Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsRP3beta'
RP3betaRecommender: Loading complete
Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:05<00:00, 5321.51it/s]


Unloading RP3beta...
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:09<00:00, 2894.20it/s]


Unloading SLIMElasticNet...


,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,NMF_Recommended,MultVAE_Score,MultVAE_RankPosition,MultVAE_Recommended,ItemKNN_tversky_MaxSim,ItemKNN_tversky_StdSim,RP3beta_MaxSim,RP3beta_StdSim,SLIMElasticNet_MaxSim,SLIMElasticNet_StdSim
UserID,,,,,,,,,,,,,,,,,,,,,
0,3607,False,0.454812,26,0,0.699813,12,0,0.585087,7,...,0,0.654270,105,0,0.000000,0.000000,0.048630,0.006384,0.041168,0.008470
0,3551,False,0.155081,313,0,0.510249,82,0,0.112332,189,...,0,0.776577,21,0,0.000000,0.000000,0.040243,0.005283,0.024707,0.006255
0,6730,False,0.366663,55,0,0.568895,48,0,0.199446,85,...,0,0.714261,52,0,0.268439,0.035242,0.033195,0.004358,0.044206,0.006832
0,6492,False,0.063416,935,0,0.426659,144,0,0.106174,201,...,0,0.695365,67,0,0.262334,0.034441,0.068140,0.008946,0.194642,0.025609
0,3754,False,0.072911,818,0,0.386167,194,0,0.150095,135,...,0,0.594878,154,0,0.411530,0.054028,0.088523,0.011622,0.287283,0.037972
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27094,6330,False,0.410620,35,0,0.048795,1800,0,0.000000,4711,...,0,-0.149982,3571,0,0.000000,0.000000,0.000000,0.000000,0.015519,0.001484
27094,6584,False,0.412144,34,0,0.104053,997,0,0.000000,5823,...,0,0.000014,2513,0,0.000000,0.000000,0.000000,0.000000,0.021471,0.002021
27094,2369,False,0.428672,29,0,0.092265,1110,0,0.000000,2664,...,0,0.012532,2433,0,0.000000,0.000000,0.000000,0.000000,0.014133,0.001788


In [22]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Embedding Features**

In [23]:
training_dataframe = add_embedding_features_batched(training_dataframe, URM_train, "train_models")
training_dataframe

Resetting index to restore UserID column...
Loading IALS model...
IALSRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/train_modelsIALS.zip'
IALSRecommender: Loading complete
Calculating K-Means Clusters...
Fitting PCA on random sample...
Processing Batches (Batch Size: 200000)...


100%|██████████| 34/34 [00:03<00:00,  8.51it/s]


Embedding features added successfully.


,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,...,SLIMElasticNet_MaxSim,SLIMElasticNet_StdSim,User_Cluster,Item_Cluster,IALS_EuclideanDist,IALS_PCA_0,IALS_PCA_1,IALS_PCA_2,IALS_PCA_3,IALS_PCA_4
0,0,3607,False,0.454812,26,0,0.699813,12,0,0.585087,...,0.041168,0.008470,8,8,3.950507,-0.041958,-0.077987,0.076606,0.019783,0.038072
1,0,3551,False,0.155081,313,0,0.510249,82,0,0.112332,...,0.024707,0.006255,8,8,4.064815,-0.057691,-0.032119,0.025795,0.025127,-0.117859
2,0,6730,False,0.366663,55,0,0.568895,48,0,0.199446,...,0.044206,0.006832,8,8,4.032704,0.046279,-0.120728,0.020239,0.006101,0.085546
3,0,6492,False,0.063416,935,0,0.426659,144,0,0.106174,...,0.194642,0.025609,8,2,3.895789,-0.003278,-0.083382,0.013714,0.073172,0.016321
4,0,3754,False,0.072911,818,0,0.386167,194,0,0.150095,...,0.287283,0.037972,8,2,3.964033,0.004504,-0.044802,0.054064,0.054716,0.000371
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6601559,27094,6330,False,0.410620,35,0,0.048795,1800,0,0.000000,...,0.015519,0.001484,2,5,5.700868,-0.071384,-0.022635,-0.083270,0.077368,0.027988
6601560,27094,6584,False,0.412144,34,0,0.104053,997,0,0.000000,...,0.021471,0.002021,2,4,5.702462,-0.143660,-0.004827,-0.096247,0.038163,-0.004074
6601561,27094,2369,False,0.428672,29,0,0.092265,1110,0,0.000000,...,0.014133,0.001788,2,4,5.740954,-0.195302,-0.029825,-0.056283,0.113549,-0.065434
6601562,27094,3316,False,0.424921,30,0,0.050882,1757,0,0.000000,...,0.009062,0.001175,2,5,5.656553,-0.174920,-0.000963,-0.144922,0.087853,0.013606


In [24]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Aggregate Features**

In [25]:
training_dataframe = add_aggregate_features_stats(training_dataframe)
training_dataframe

,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,...,IALS_PCA_4,Counter_Recommended,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score
0,0,3607,False,0.454812,26,0,0.699813,12,0,0.585087,...,0.038072,4,735.818182,1819.599528,2.369416,4.159177,0.446466,0.249019,-0.409624,0.469747
1,0,3551,False,0.155081,313,0,0.510249,82,0,0.112332,...,-0.117859,0,1160.545455,2135.449696,1.730906,1.235297,0.271795,0.268083,1.059486,0.943434
2,0,6730,False,0.366663,55,0,0.568895,48,0,0.199446,...,0.085546,0,796.909091,1570.505800,1.967968,2.312954,0.311131,0.255362,1.059846,0.727341
3,0,6492,False,0.063416,935,0,0.426659,144,0,0.106174,...,0.016321,0,339.090909,705.046632,4.079092,17.686486,0.247482,0.224877,1.980924,4.484738
4,0,3754,False,0.072911,818,0,0.386167,194,0,0.150095,...,0.000371,0,316.909091,495.750889,2.607854,6.882978,0.250314,0.190596,2.645183,8.047985
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6601559,27094,6330,False,0.410620,35,0,0.048795,1800,0,0.000000,...,0.027988,0,2856.590909,2487.666312,0.326660,-1.709579,0.065229,0.239922,2.278579,7.712372
6601560,27094,6584,False,0.412144,34,0,0.104053,997,0,0.000000,...,-0.004074,0,2631.454545,2361.129587,0.574879,-1.441451,0.090876,0.237749,2.082838,6.828017
6601561,27094,2369,False,0.428672,29,0,0.092265,1110,0,0.000000,...,-0.065434,0,2027.136364,2138.303941,1.402101,1.010271,0.078785,0.258349,1.374087,5.300080
6601562,27094,3316,False,0.424921,30,0,0.050882,1757,0,0.000000,...,0.013606,0,2770.409091,2543.396130,0.605881,-1.533561,0.073155,0.253527,1.555364,6.181817


In [29]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **User Stats**

In [30]:
training_dataframe = add_user_stats(training_dataframe)
training_dataframe

,UserID,ItemID,Label,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,...,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,3607,False,0.454812,26,0,0.699813,12,0,0.585087,...,735.818176,1819.599487,2.369416,4.159177,0.446466,0.249019,-0.409624,0.469747,57,3858
1,0,3551,False,0.155081,313,0,0.510249,82,0,0.112332,...,1160.545410,2135.449707,1.730906,1.235297,0.271795,0.268083,1.059486,0.943434,57,1314
2,0,6730,False,0.366663,55,0,0.568895,48,0,0.199446,...,796.909119,1570.505859,1.967968,2.312954,0.311131,0.255362,1.059846,0.727341,57,3155
3,0,6492,False,0.063416,935,0,0.426659,144,0,0.106174,...,339.090912,705.046631,4.079092,17.686485,0.247482,0.224877,1.980924,4.484738,57,516
4,0,3754,False,0.072911,818,0,0.386167,194,0,0.150095,...,316.909088,495.750885,2.607854,6.882978,0.250314,0.190596,2.645183,8.047985,57,606
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6601559,27094,6330,False,0.410620,35,0,0.048795,1800,0,0.000000,...,2856.590820,2487.666260,0.326660,-1.709579,0.065229,0.239922,2.278579,7.712372,205,3501
6601560,27094,6584,False,0.412144,34,0,0.104053,997,0,0.000000,...,2631.454590,2361.129639,0.574879,-1.441451,0.090876,0.237749,2.082838,6.828017,205,3522
6601561,27094,2369,False,0.428672,29,0,0.092265,1110,0,0.000000,...,2027.136353,2138.303955,1.402101,1.010271,0.078785,0.258349,1.374087,5.300080,205,3588
6601562,27094,3316,False,0.424921,30,0,0.050882,1757,0,0.000000,...,2770.409180,2543.396240,0.605881,-1.533561,0.073155,0.253527,1.555364,6.181817,205,3667


In [28]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Sanity check**

In [31]:
sanity_check(training_dataframe)

--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

[WARNING] UserID is int16, expected int. (Did a merge fail?)

[WARNING] ItemID is int16, expected int.

--- SANITY CHECK PASSED: Data is clean ---


True

### **Save Dataframe**

In [32]:
# Parquet handles indices, but it's safer/cleaner to reset it 
# so 'UserID' is a normal column and not a special index.
if training_dataframe.index.name == 'UserID':
    training_dataframe = training_dataframe.reset_index()

# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "training_data.parquet")

training_dataframe.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='zstd',
    index=False
)

print(f"Saved successfully to: {save_path}")

Saved successfully to: /home/luigi/RecSys/xg_boost/dataframes/training_data.parquet


In [33]:
del training_dataframe
gc.collect()

20

## **Validation Dataframe**

### **Generate Candidates**

In [15]:
training_dataframe = generate_candidates(URM_train_complete, "validation_candidates")
training_dataframe

Model found: SLIMElasticNet
Model found: ItemKNN_tversky
Model found: RP3beta
Model found: TopPop
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_candidatesSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Generating candidates for model: SLIMElasticNet
Unloading SLIMElasticNet...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_candidatesItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Generating candidates for model: ItemKNN_tversky
Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_candidatesRP3beta'
RP3betaRecommender: Loading complete
Generating candidates for model: RP3beta
Unloading RP3beta...
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_candidatesTopPop'
TopPopRecommen

,UserID,ItemID
0,0,2530
1,0,2392
2,0,4264
3,0,4486
4,0,828
...,...,...
10837991,27094,1038
10837993,27094,6382
10837994,27094,4796
10837998,27094,1217


### **Models Features**

In [16]:
training_dataframe = add_models_features(training_dataframe, URM_train_complete, "validation_models")
training_dataframe

Model found: TopPop
Model found: ItemKNN_cosine
Model found: ItemKNN_jaccard
Model found: ItemKNN_asymmetric
Model found: ItemKNN_tversky
Model found: ItemKNN_dice
Model found: UserKNN_cosine
Model found: UserKNN_jaccard
Model found: UserKNN_asymmetric
Model found: UserKNN_tversky
Model found: UserKNN_dice
Model found: SLIMElasticNet
Model found: EASE_R
Model found: P3alpha
Model found: RP3beta
Model found: IALS
Model found: MatrixFactorization_WARP
Model found: MatrixFactorization_BPR
Model found: MatrixFactorization_SVDpp
Model found: SLIM_BPR
Model found: NMF
Model found: MultVAE
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsTopPop'
TopPopRecommender: Loading complete
Processing features for model: TopPop


Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_cosine


Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_jaccard


Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsItemKNN_asymmetric'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_asymmetric


Unloading ItemKNN_asymmetric...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_tversky


...

Unloading SLIM_BPR...
Loading NMF...
NMFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsNMF'
NMFRecommender: Loading complete
Processing features for model: NMF


Unloading NMF...
Loading MultVAE...
Processing features for model: MultVAE


Batches MultVAE:   0%|          | 0/55 [00:00<?, ?it/s]/home/luigi/RecSys/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:203: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(


Unloading MultVAE...


,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,MatrixFactorization_SVDpp_Recommended,SLIM_BPR_Score,SLIM_BPR_RankPosition,SLIM_BPR_Recommended,NMF_Score,NMF_RankPosition,NMF_Recommended,MultVAE_Score,MultVAE_RankPosition,MultVAE_Recommended
0,0,2530,0.024162,1902,0,0.440498,106,0,0.357339,40,...,0,0.028101,501,0,3.354477e-02,290,0,0.352991,440,0
1,0,3754,0.071174,822,0,0.347369,216,0,0.172276,154,...,0,0.074413,323,0,1.363868e-05,4471,0,0.523851,151,0
2,0,6190,0.138415,361,0,0.575431,28,0,0.416094,28,...,0,0.231997,147,0,8.144859e-02,98,0,0.545478,132,0
3,0,6492,0.061341,949,0,0.385426,160,0,0.167151,163,...,0,0.030959,481,0,7.059702e-02,115,0,0.657774,34,0
4,0,1458,0.378629,49,0,0.619288,20,0,0.672616,9,...,0,0.603289,14,0,1.518368e-01,36,0,0.637639,58,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6470038,27094,1418,0.380970,45,0,0.063481,1599,0,0.000000,4068,...,0,0.152963,156,0,1.668941e-14,5480,0,-0.146873,3276,0
6470039,27094,2779,0.370388,47,0,0.000000,5138,0,0.000000,6077,...,0,0.125899,188,0,3.592389e-03,2236,0,-0.103086,2964,0
6470040,27094,4180,0.367297,48,0,0.042708,2024,0,0.000000,4855,...,0,0.144312,165,0,2.420992e-03,2470,0,-0.181220,3524,0
6470041,27094,4790,0.022289,1929,0,0.462146,71,0,0.215428,154,...,0,0.020522,494,0,2.201255e-01,69,0,0.727176,22,0


In [17]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Item Similarity Features**

In [18]:
training_dataframe = calculate_item_item_features_fast(training_dataframe, URM_train_complete, "validation_models")
training_dataframe

Model found: ItemKNN_tversky
Model found: RP3beta
Model found: SLIMElasticNet
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Extracting MAX similarity for ItemKNN_tversky...


100%|██████████| 27095/27095 [00:04<00:00, 5957.61it/s]


Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsRP3beta'
RP3betaRecommender: Loading complete
Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:04<00:00, 5770.87it/s]


Unloading RP3beta...
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:09<00:00, 2879.02it/s]


Unloading SLIMElasticNet...


,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,ItemKNN_jaccard_Recommended,...,NMF_Recommended,MultVAE_Score,MultVAE_RankPosition,MultVAE_Recommended,ItemKNN_tversky_MaxSim,ItemKNN_tversky_StdSim,RP3beta_MaxSim,RP3beta_StdSim,SLIMElasticNet_MaxSim,SLIMElasticNet_StdSim
UserID,,,,,,,,,,,,,,,,,,,,,
0,2530,0.024162,1902,0,0.440498,106,0,0.357339,40,0,...,0,0.352991,440,0,0.304353,0.048236,0.085056,0.013395,0.236689,0.031940
0,3754,0.071174,822,0,0.347369,216,0,0.172276,154,0,...,0,0.523851,151,0,0.521651,0.057957,0.090152,0.010016,0.366841,0.040989
0,6190,0.138415,361,0,0.575431,28,0,0.416094,28,0,...,0,0.545478,132,0,0.481924,0.074700,0.074147,0.011038,0.197584,0.027081
0,6492,0.061341,949,0,0.385426,160,0,0.167151,163,0,...,0,0.657774,34,0,0.342031,0.038000,0.070411,0.007823,0.254138,0.028505
0,1458,0.378629,49,0,0.619288,20,0,0.672616,9,1,...,0,0.637639,58,0,0.000000,0.000000,0.047448,0.005272,0.040835,0.007305
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27094,1418,0.380970,45,0,0.063481,1599,0,0.000000,4068,0,...,0,-0.146873,3276,0,0.000000,0.000000,0.000000,0.000000,0.018880,0.001631
27094,2779,0.370388,47,0,0.000000,5138,0,0.000000,6077,0,...,0,-0.103086,2964,0,0.000000,0.000000,0.000000,0.000000,0.015284,0.001496
27094,4180,0.367297,48,0,0.042708,2024,0,0.000000,4855,0,...,0,-0.181220,3524,0,0.000000,0.000000,0.000000,0.000000,0.026477,0.002307


In [19]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Embedding Features**

In [20]:
training_dataframe = add_embedding_features_batched(training_dataframe, URM_train_complete, "validation_models")
training_dataframe

Resetting index to restore UserID column...
Loading IALS model...
IALSRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/validation_modelsIALS.zip'
IALSRecommender: Loading complete
Calculating K-Means Clusters...
Fitting PCA on random sample...
Processing Batches (Batch Size: 200000)...


100%|██████████| 33/33 [00:03<00:00,  8.79it/s]


Embedding features added successfully.


,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,SLIMElasticNet_MaxSim,SLIMElasticNet_StdSim,User_Cluster,Item_Cluster,IALS_EuclideanDist,IALS_PCA_0,IALS_PCA_1,IALS_PCA_2,IALS_PCA_3,IALS_PCA_4
0,0,2530,0.024162,1902,0,0.440498,106,0,0.357339,40,...,0.236689,0.031940,4,8,4.202888,-0.017865,0.036761,-0.002758,-0.022733,-0.004176
1,0,3754,0.071174,822,0,0.347369,216,0,0.172276,154,...,0.366841,0.040989,4,8,4.358303,-0.031022,-0.006967,0.041083,-0.055945,-0.012833
2,0,6190,0.138415,361,0,0.575431,28,0,0.416094,28,...,0.197584,0.027081,4,7,4.307730,-0.031571,0.018036,0.038078,-0.040295,0.015728
3,0,6492,0.061341,949,0,0.385426,160,0,0.167151,163,...,0.254138,0.028505,4,2,4.288273,-0.040355,0.046770,0.025102,-0.057937,0.012351
4,0,1458,0.378629,49,0,0.619288,20,0,0.672616,9,...,0.040835,0.007305,4,7,4.395815,-0.025513,0.057965,0.044416,-0.083411,-0.009073
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6470038,27094,1418,0.380970,45,0,0.063481,1599,0,0.000000,4068,...,0.018880,0.001631,2,6,5.932883,-0.145781,-0.026361,0.059000,0.027536,-0.025629
6470039,27094,2779,0.370388,47,0,0.000000,5138,0,0.000000,6077,...,0.015284,0.001496,2,6,5.936064,-0.073121,-0.018391,0.115031,-0.003447,-0.069514
6470040,27094,4180,0.367297,48,0,0.042708,2024,0,0.000000,4855,...,0.026477,0.002307,2,6,5.949266,-0.071163,-0.025268,0.112736,0.095235,-0.052070
6470041,27094,4790,0.022289,1929,0,0.462146,71,0,0.215428,154,...,0.055577,0.005947,2,8,5.668177,-0.063471,-0.019380,0.083753,0.034627,-0.026548


In [21]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Aggregate Features**

In [22]:
training_dataframe = add_aggregate_features_stats(training_dataframe)
training_dataframe

,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,IALS_PCA_4,Counter_Recommended,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score
0,0,2530,0.024162,1902,0,0.440498,106,0,0.357339,40,...,-0.004176,7,338.454545,581.330307,2.212248,4.269336,0.373518,0.253683,0.509799,-0.677177
1,0,3754,0.071174,822,0,0.347369,216,0,0.172276,154,...,-0.012833,0,667.318182,1294.205672,2.455083,4.953726,0.210426,0.194799,2.039030,5.926878
2,0,6190,0.138415,361,0,0.575431,28,0,0.416094,28,...,0.015728,0,248.090909,818.899140,4.575913,21.220274,0.332196,0.209271,1.041836,2.257536
3,0,6492,0.061341,949,0,0.385426,160,0,0.167151,163,...,0.012351,0,457.727273,1253.169132,4.282579,19.140955,0.248390,0.232927,0.857162,3.276469
4,0,1458,0.378629,49,0,0.619288,20,0,0.672616,9,...,-0.009073,8,602.590909,1569.446632,3.143804,10.356739,0.387330,0.303650,-0.748551,1.461609
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6470038,27094,1418,0.380970,45,0,0.063481,1599,0,0.000000,4068,...,-0.025629,0,2327.045455,1995.589625,0.783627,-0.507307,0.081324,0.252887,1.961992,6.522962
6470039,27094,2779,0.370388,47,0,0.000000,5138,0,0.000000,6077,...,-0.069514,0,2482.454545,2093.020368,0.878198,-0.561856,0.061739,0.235025,2.613684,10.073752
6470040,27094,4180,0.367297,48,0,0.042708,2024,0,0.000000,4855,...,-0.052070,0,3230.727273,2568.178365,0.238093,-1.787520,0.067717,0.256604,2.103574,6.575085
6470041,27094,4790,0.022289,1929,0,0.462146,71,0,0.215428,154,...,-0.026548,0,247.727273,416.068582,3.527030,13.640942,0.338936,0.248854,0.853468,0.349615


In [23]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **User Stats**

In [24]:
training_dataframe = add_user_stats(training_dataframe)
training_dataframe

,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,2530,0.024162,1902,0,0.440498,106,0,0.357339,40,...,338.454559,581.330322,2.212248,4.269335,0.373518,0.253683,0.509799,-0.677177,70,207
1,0,3754,0.071174,822,0,0.347369,216,0,0.172276,154,...,667.318176,1294.205688,2.455083,4.953726,0.210426,0.194799,2.039030,5.926878,70,594
2,0,6190,0.138415,361,0,0.575431,28,0,0.416094,28,...,248.090912,818.899170,4.575913,21.220274,0.332196,0.209271,1.041836,2.257536,70,1179
3,0,6492,0.061341,949,0,0.385426,160,0,0.167151,163,...,457.727264,1253.169189,4.282579,19.140955,0.248390,0.232927,0.857162,3.276469,70,531
4,0,1458,0.378629,49,0,0.619288,20,0,0.672616,9,...,602.590881,1569.446655,3.143804,10.356739,0.387330,0.303650,-0.748551,1.461609,70,3214
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6470038,27094,1418,0.380970,45,0,0.063481,1599,0,0.000000,4068,...,2327.045410,1995.589600,0.783627,-0.507307,0.081324,0.252887,1.961992,6.522962,192,3313
6470039,27094,2779,0.370388,47,0,0.000000,5138,0,0.000000,6077,...,2482.454590,2093.020264,0.878198,-0.561856,0.061739,0.235025,2.613684,10.073752,192,3161
6470040,27094,4180,0.367297,48,0,0.042708,2024,0,0.000000,4855,...,3230.727295,2568.178467,0.238093,-1.787520,0.067717,0.256604,2.103574,6.575085,192,3110
6470041,27094,4790,0.022289,1929,0,0.462146,71,0,0.215428,154,...,247.727280,416.068573,3.527030,13.640942,0.338936,0.248854,0.853468,0.349615,192,195


In [25]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Sanity Check**

In [26]:
sanity_check(training_dataframe)

--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

[WARNING] UserID is int16, expected int. (Did a merge fail?)

[WARNING] ItemID is int16, expected int.

--- SANITY CHECK PASSED: Data is clean ---


True

### **Save Dataframe**

In [27]:
# Parquet handles indices, but it's safer/cleaner to reset it 
# so 'UserID' is a normal column and not a special index.
if training_dataframe.index.name == 'UserID':
    training_dataframe = training_dataframe.reset_index()

# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "validation_data.parquet")

training_dataframe.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='zstd',
    index=False
)

print(f"Saved successfully to: {save_path}")

Saved successfully to: /home/luigi/RecSys/xg_boost/dataframes/validation_data.parquet


In [28]:
del training_dataframe
gc.collect()

20

## **Prediction Dataframe**

In [15]:
all_data = URM_train_complete + URM_test

### **Generate Candidates**

In [16]:
training_dataframe = generate_candidates(all_data, "pred_candidates")
training_dataframe

Model found: SLIMElasticNet
Model found: ItemKNN_tversky
Model found: RP3beta
Model found: TopPop
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_candidatesSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Generating candidates for model: SLIMElasticNet
Unloading SLIMElasticNet...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_candidatesItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Generating candidates for model: ItemKNN_tversky
Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_candidatesRP3beta'
RP3betaRecommender: Loading complete
Generating candidates for model: RP3beta
Unloading RP3beta...
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_candidatesTopPop'
TopPopRecommender: Loading complete
Ge

,UserID,ItemID
0,0,2530
1,0,2392
2,0,6411
3,0,5532
4,0,4486
...,...,...
10837995,27094,4625
10837996,27094,291
10837997,27094,1403
10837998,27094,5356


### **Models Features**

In [17]:
training_dataframe = add_models_features(training_dataframe, all_data, "pred_models")
training_dataframe

Model found: TopPop
Model found: ItemKNN_cosine
Model found: ItemKNN_jaccard
Model found: ItemKNN_asymmetric
Model found: ItemKNN_tversky
Model found: ItemKNN_dice
Model found: UserKNN_cosine
Model found: UserKNN_jaccard
Model found: UserKNN_asymmetric
Model found: UserKNN_tversky
Model found: UserKNN_dice
Model found: SLIMElasticNet
Model found: EASE_R
Model found: P3alpha
Model found: RP3beta
Model found: IALS
Model found: MatrixFactorization_WARP
Model found: MatrixFactorization_BPR
Model found: MatrixFactorization_SVDpp
Model found: SLIM_BPR
Model found: NMF
Model found: MultVAE
Loading TopPop...
TopPopRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsTopPop'
TopPopRecommender: Loading complete
Processing features for model: TopPop


Unloading TopPop...
Loading ItemKNN_cosine...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsItemKNN_cosine'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_cosine


Unloading ItemKNN_cosine...
Loading ItemKNN_jaccard...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsItemKNN_jaccard'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_jaccard


Unloading ItemKNN_jaccard...
Loading ItemKNN_asymmetric...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsItemKNN_asymmetric'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_asymmetric


Unloading ItemKNN_asymmetric...
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Processing features for model: ItemKNN_tversky


...

Unloading SLIM_BPR...
Loading NMF...
NMFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsNMF'
NMFRecommender: Loading complete
Processing features for model: NMF


Unloading NMF...
Loading MultVAE...
Processing features for model: MultVAE


Batches MultVAE:   0%|          | 0/55 [00:00<?, ?it/s]/home/luigi/RecSys/Recommenders/Neural/MultVAE_PyTorch_Recommender.py:203: UserWarning: Sparse CSR tensor support is in beta state. If you miss a functionality in the sparse tensor support, please submit a feature request to https://github.com/pytorch/pytorch/issues. (Triggered internally at /pytorch/aten/src/ATen/SparseCsrTensorImpl.cpp:53.)
  user_batch_tensor = torch.sparse_csr_tensor(


Unloading MultVAE...


,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,MatrixFactorization_SVDpp_Recommended,SLIM_BPR_Score,SLIM_BPR_RankPosition,SLIM_BPR_Recommended,NMF_Score,NMF_RankPosition,NMF_Recommended,MultVAE_Score,MultVAE_RankPosition,MultVAE_Recommended
0,0,2530,0.025039,1859,0,0.486296,69,0,0.335175,42,...,0,0.026231,521,0,7.973564e-02,101,0,0.382079,384,0
1,0,2392,0.029852,1656,0,0.477561,74,0,0.429715,21,...,0,0.042008,430,0,8.212388e-02,93,0,0.454756,236,0
2,0,6411,0.079329,739,0,0.708008,4,1,0.819056,2,...,0,0.217932,150,0,3.804802e-01,1,1,0.674653,17,0
3,0,647,0.034815,1478,0,0.129285,966,0,0.031515,694,...,0,0.002224,1048,0,1.478461e-02,734,0,-0.005758,1950,0
4,0,6190,0.138883,352,0,0.512421,57,0,0.376496,33,...,0,0.217138,151,0,1.269505e-01,46,0,0.562460,102,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6481192,27094,5662,0.267840,99,0,0.016543,2976,0,0.000000,2736,...,0,0.075198,253,0,1.242545e-09,4622,0,-0.296636,4329,0
6481193,27094,2232,0.324611,62,0,0.066327,1561,0,0.000000,3153,...,0,0.158204,144,0,0.000000e+00,6646,0,-0.162489,3416,0
6481194,27094,2085,0.020678,1991,0,0.308810,223,0,0.254972,85,...,0,0.024474,435,0,1.788581e-01,93,0,0.666188,72,0
6481195,27094,5068,0.010302,2793,0,0.211153,478,0,0.160868,193,...,0,0.011577,607,0,1.154982e-01,175,0,0.476609,336,0


In [18]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Item Similarity Features**

In [19]:
training_dataframe = calculate_item_item_features_fast(training_dataframe, all_data, "pred_models")
training_dataframe

Model found: ItemKNN_tversky
Model found: RP3beta
Model found: SLIMElasticNet
Loading ItemKNN_tversky...
ItemKNNCFRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsItemKNN_tversky'
ItemKNNCFRecommender: Loading complete
Extracting MAX similarity for ItemKNN_tversky...


100%|██████████| 27095/27095 [00:04<00:00, 5732.33it/s]


Unloading ItemKNN_tversky...
Loading RP3beta...
RP3betaRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsRP3beta'
RP3betaRecommender: Loading complete
Extracting MAX similarity for RP3beta...


100%|██████████| 27095/27095 [00:05<00:00, 5412.54it/s]


Unloading RP3beta...
Loading SLIMElasticNet...
SLIMElasticNetRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsSLIMElasticNet'
SLIMElasticNetRecommender: Loading complete
Extracting MAX similarity for SLIMElasticNet...


100%|██████████| 27095/27095 [00:10<00:00, 2557.67it/s]


Unloading SLIMElasticNet...


,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,ItemKNN_jaccard_Recommended,...,NMF_Recommended,MultVAE_Score,MultVAE_RankPosition,MultVAE_Recommended,ItemKNN_tversky_MaxSim,ItemKNN_tversky_StdSim,RP3beta_MaxSim,RP3beta_StdSim,SLIMElasticNet_MaxSim,SLIMElasticNet_StdSim
UserID,,,,,,,,,,,,,,,,,,,,,
0,2530,0.025039,1859,0,0.486296,69,0,0.335175,42,0,...,0,0.382079,384,0,0.378028,0.057963,0.082162,0.012472,0.268827,0.033691
0,2392,0.029852,1656,0,0.477561,74,0,0.429715,21,0,...,0,0.454756,236,0,0.537423,0.076996,0.103341,0.013129,0.510629,0.053192
0,6411,0.079329,739,0,0.708008,4,1,0.819056,2,1,...,1,0.674653,17,0,0.650149,0.124703,0.081791,0.015166,0.288247,0.038867
0,647,0.034815,1478,0,0.129285,966,0,0.031515,694,0,...,0,-0.005758,1950,0,0.107528,0.010917,0.034426,0.003495,0.085708,0.008960
0,6190,0.138883,352,0,0.512421,57,0,0.376496,33,0,...,0,0.562460,102,0,0.623967,0.088268,0.074935,0.010294,0.256107,0.030168
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27094,5662,0.267840,99,0,0.016543,2976,0,0.000000,2736,0,...,0,-0.296636,4329,0,0.000000,0.000000,0.000000,0.000000,0.028027,0.002263
27094,2232,0.324611,62,0,0.066327,1561,0,0.000000,3153,0,...,0,-0.162489,3416,0,0.000000,0.000000,0.000000,0.000000,0.011038,0.001555
27094,2085,0.020678,1991,0,0.308810,223,0,0.254972,85,0,...,0,0.666188,72,0,0.316227,0.017817,0.040957,0.005300,0.124242,0.008629


In [20]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Embedding Features**

In [21]:
training_dataframe = add_embedding_features_batched(training_dataframe, all_data, "pred_models")
training_dataframe

Resetting index to restore UserID column...
Loading IALS model...
IALSRecommender: Loading model from file '/home/luigi/RecSys/xg_boost/models/pred_modelsIALS.zip'
IALSRecommender: Loading complete
Calculating K-Means Clusters...
Fitting PCA on random sample...
Processing Batches (Batch Size: 200000)...


100%|██████████| 33/33 [00:03<00:00,  8.47it/s]


Embedding features added successfully.


,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,SLIMElasticNet_MaxSim,SLIMElasticNet_StdSim,User_Cluster,Item_Cluster,IALS_EuclideanDist,IALS_PCA_0,IALS_PCA_1,IALS_PCA_2,IALS_PCA_3,IALS_PCA_4
0,0,2530,0.025039,1859,0,0.486296,69,0,0.335175,42,...,0.268827,0.033691,1,6,4.229155,-0.010706,-0.042214,-0.007174,-0.018552,-0.028505
1,0,2392,0.029852,1656,0,0.477561,74,0,0.429715,21,...,0.510629,0.053192,1,7,4.286417,-0.045390,-0.033853,-0.007466,-0.014718,-0.003266
2,0,6411,0.079329,739,0,0.708008,4,1,0.819056,2,...,0.288247,0.038867,1,7,4.256270,0.003057,-0.042773,0.004220,-0.030066,-0.004079
3,0,647,0.034815,1478,0,0.129285,966,0,0.031515,694,...,0.085708,0.008960,1,7,4.424312,-0.044441,-0.032640,-0.002029,-0.022176,-0.018452
4,0,6190,0.138883,352,0,0.512421,57,0,0.376496,33,...,0.256107,0.030168,1,7,4.343050,-0.039542,-0.071416,-0.033198,0.005166,0.007417
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6481192,27094,5662,0.267840,99,0,0.016543,2976,0,0.000000,2736,...,0.028027,0.002263,4,2,6.078342,-0.119018,0.036011,-0.072204,-0.033763,0.027122
6481193,27094,2232,0.324611,62,0,0.066327,1561,0,0.000000,3153,...,0.011038,0.001555,4,2,6.059484,-0.161602,0.057796,-0.098926,-0.079939,0.021888
6481194,27094,2085,0.020678,1991,0,0.308810,223,0,0.254972,85,...,0.124242,0.008629,4,6,5.896640,-0.103615,-0.020765,-0.008332,-0.033563,0.016841
6481195,27094,5068,0.010302,2793,0,0.211153,478,0,0.160868,193,...,0.038229,0.002247,4,6,5.920374,-0.077109,-0.018042,-0.013472,-0.041423,0.003712


In [22]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Aggregate Features**

In [23]:
training_dataframe = add_aggregate_features_stats(training_dataframe)
training_dataframe

,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,IALS_PCA_4,Counter_Recommended,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score
0,0,2530,0.025039,1859,0,0.486296,69,0,0.335175,42,...,-0.028505,6,268.681818,493.435033,2.359348,5.294017,0.397710,0.244865,0.565020,-0.123574
1,0,2392,0.029852,1656,0,0.477561,74,0,0.429715,21,...,-0.003266,5,475.500000,1266.180110,4.054056,17.426017,0.306729,0.245669,0.242043,1.594079
2,0,6411,0.079329,739,0,0.708008,4,1,0.819056,2,...,-0.004079,16,197.772727,629.295595,4.165599,18.186410,0.535907,0.217172,-0.656013,0.912273
3,0,647,0.034815,1478,0,0.129285,966,0,0.031515,694,...,-0.018452,0,1049.409091,705.936437,-0.054673,-1.119495,0.100246,0.194093,3.997774,17.285295
4,0,6190,0.138883,352,0,0.512421,57,0,0.376496,33,...,0.007417,0,290.727273,747.111529,3.373701,11.320082,0.315294,0.194093,1.414928,3.192074
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6481192,27094,5662,0.267840,99,0,0.016543,2976,0,0.000000,2736,...,0.027122,0,3318.818182,1994.473355,0.019928,-1.271042,0.052173,0.235991,2.275996,8.543265
6481193,27094,2232,0.324611,62,0,0.066327,1561,0,0.000000,3153,...,0.021888,0,2608.045455,1841.860175,0.963881,0.615430,0.070834,0.218185,3.340281,13.085633
6481194,27094,2085,0.020678,1991,0,0.308810,223,0,0.254972,85,...,0.016841,0,515.772727,794.571698,2.694688,7.524818,0.232067,0.219507,1.980587,4.282604
6481195,27094,5068,0.010302,2793,0,0.211153,478,0,0.160868,193,...,0.003712,0,785.090909,807.634636,2.177368,4.588331,0.174733,0.197847,2.461800,7.123139


In [24]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **User Stats**

In [25]:
training_dataframe = add_user_stats(training_dataframe)
training_dataframe

,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,2530,0.025039,1859,0,0.486296,69,0,0.335175,42,...,268.681824,493.435028,2.359348,5.294017,0.397710,0.244865,0.565020,-0.123574,61,205
1,0,2392,0.029852,1656,0,0.477561,74,0,0.429715,21,...,475.500000,1266.180054,4.054056,17.426016,0.306729,0.245669,0.242043,1.594079,61,250
2,0,6411,0.079329,739,0,0.708008,4,1,0.819056,2,...,197.772720,629.295593,4.165599,18.186411,0.535907,0.217172,-0.656013,0.912273,61,683
3,0,647,0.034815,1478,0,0.129285,966,0,0.031515,694,...,1049.409058,705.936462,-0.054673,-1.119495,0.100246,0.194093,3.997774,17.285295,61,299
4,0,6190,0.138883,352,0,0.512421,57,0,0.376496,33,...,290.727264,747.111511,3.373701,11.320082,0.315294,0.194093,1.414928,3.192074,61,1169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6481192,27094,5662,0.267840,99,0,0.016543,2976,0,0.000000,2736,...,3318.818115,1994.473389,0.019928,-1.271042,0.052173,0.235991,2.275996,8.543265,200,2267
6481193,27094,2232,0.324611,62,0,0.066327,1561,0,0.000000,3153,...,2608.045410,1841.860229,0.963881,0.615430,0.070834,0.218185,3.340281,13.085633,200,2744
6481194,27094,2085,0.020678,1991,0,0.308810,223,0,0.254972,85,...,515.772705,794.571716,2.694688,7.524817,0.232067,0.219507,1.980587,4.282604,200,166
6481195,27094,5068,0.010302,2793,0,0.211153,478,0,0.160868,193,...,785.090881,807.634644,2.177368,4.588331,0.174733,0.197847,2.461800,7.123139,200,87


In [26]:
training_dataframe = optimize_dataframe_types(training_dataframe)
gc.collect()

0

### **Sanity Check**

In [27]:
sanity_check(training_dataframe)

--- STARTING SANITY CHECK ---
[OK] No NaNs found.
[OK] No Infinite values found.
[OK] Keys (UserID, ItemID) are unique.

[WARNING] UserID is int16, expected int. (Did a merge fail?)

[WARNING] ItemID is int16, expected int.

--- SANITY CHECK PASSED: Data is clean ---


True

### **Save Dataframe**

In [28]:
training_dataframe

,UserID,ItemID,TopPop_Score,TopPop_RankPosition,TopPop_Recommended,ItemKNN_cosine_Score,ItemKNN_cosine_RankPosition,ItemKNN_cosine_Recommended,ItemKNN_jaccard_Score,ItemKNN_jaccard_RankPosition,...,Mean_RankPosition,Std_RankPosition,Skew_RankPosition,Kurtosis_RankPosition,Mean_Score,Std_Score,Skew_Score,Kurtosis_Score,User_Profile_Len,Item_Global_Popularity
0,0,2530,0.025039,1859,0,0.486296,69,0,0.335175,42,...,268.681824,493.435028,2.359348,5.294017,0.397710,0.244865,0.565020,-0.123574,61,205
1,0,2392,0.029852,1656,0,0.477561,74,0,0.429715,21,...,475.500000,1266.180054,4.054056,17.426016,0.306729,0.245669,0.242043,1.594079,61,250
2,0,6411,0.079329,739,0,0.708008,4,1,0.819056,2,...,197.772720,629.295593,4.165599,18.186411,0.535907,0.217172,-0.656013,0.912273,61,683
3,0,647,0.034815,1478,0,0.129285,966,0,0.031515,694,...,1049.409058,705.936462,-0.054673,-1.119495,0.100246,0.194093,3.997774,17.285295,61,299
4,0,6190,0.138883,352,0,0.512421,57,0,0.376496,33,...,290.727264,747.111511,3.373701,11.320082,0.315294,0.194093,1.414928,3.192074,61,1169
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6481192,27094,5662,0.267840,99,0,0.016543,2976,0,0.000000,2736,...,3318.818115,1994.473389,0.019928,-1.271042,0.052173,0.235991,2.275996,8.543265,200,2267
6481193,27094,2232,0.324611,62,0,0.066327,1561,0,0.000000,3153,...,2608.045410,1841.860229,0.963881,0.615430,0.070834,0.218185,3.340281,13.085633,200,2744
6481194,27094,2085,0.020678,1991,0,0.308810,223,0,0.254972,85,...,515.772705,794.571716,2.694688,7.524817,0.232067,0.219507,1.980587,4.282604,200,166
6481195,27094,5068,0.010302,2793,0,0.211153,478,0,0.160868,193,...,785.090881,807.634644,2.177368,4.588331,0.174733,0.197847,2.461800,7.123139,200,87


In [29]:
# Parquet handles indices, but it's safer/cleaner to reset it 
# so 'UserID' is a normal column and not a special index.
if training_dataframe.index.name == 'UserID':
    training_dataframe = training_dataframe.reset_index()

# Save
save_path = os.path.join(XGBOOST_DATAFRAMES, "prediction_data.parquet")

training_dataframe.to_parquet(
    path=save_path,
    engine='fastparquet',
    compression='zstd',
    index=False
)

print(f"Saved successfully to: {save_path}")

Saved successfully to: /home/luigi/RecSys/xg_boost/dataframes/prediction_data.parquet


In [30]:
del training_dataframe
gc.collect()

20